In [ ]:
import pandas as pd
import random
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs

In [ ]:


def generate_negative_samples(df):
    print(f"Starting Negative Sampling for {len(df)} positive interactions...")
    
    # 1. Create the Global Positive Set (Fast Lookups)
    positive_edges = set(zip(df['UniProt_ID'], df['Single_SMILES']))
    
    # 2. Extract Unique Ligands and their Splits
    # A ligand's split is determined by its L_Cluster. We map it safely.
    ligand_splits = {}
    for _, row in df.iterrows():
        s = row['Single_SMILES']
        quadrant = row['Dataset_Split']
        # If the quadrant involves a train ligand, it's 'train'. Otherwise 'test'.
        l_split = 'train' if quadrant in ['strict_train', 'cold_target'] else 'test'
        ligand_splits[s] = l_split
        
    unique_ligands = list(ligand_splits.keys())
    train_pool = [s for s, split in ligand_splits.items() if split == 'train']
    test_pool = [s for s, split in ligand_splits.items() if split == 'test']
    
    print(f"Ligand Pools -> Train: {len(train_pool)} | Test: {len(test_pool)}")
    
    # 3. Pre-compute the Tanimoto Similarity Matrix
    print("Pre-computing Tanimoto Similarity Matrix for all 1,106 ligands...")
    mols = [Chem.MolFromSmiles(s) for s in unique_ligands]
    fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) for m in mols]
    
    # Build a dictionary of dictionaries for fast lookup: sim_matrix[smiles_A][smiles_B] = float
    sim_matrix = {s: {} for s in unique_ligands}
    for i in range(len(unique_ligands)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps)
        for j, sim in enumerate(sims):
            sim_matrix[unique_ligands[i]][unique_ligands[j]] = sim

    # 4. Generate Negatives
    negative_records = []
    hard_negative_count = 0
    easy_negative_count = 0
    
    # Convert dataframe to dict records for fast iteration
    records = df.to_dict('records')
    
    for row in records:
        p_id = row['UniProt_ID']
        true_ligand = row['Single_SMILES']
        quadrant = row['Dataset_Split']
        
        # Determine which pool to draw from based on the quadrant
        target_pool = train_pool if quadrant in ['strict_train', 'cold_target'] else test_pool
        
        # Shuffle pool to ensure random selection among valid candidates
        random.shuffle(target_pool)
        
        selected_negative = None
        is_hard = False
        
        # Pass 1: Hunt for a Hard Negative (0.3 < similarity < 0.8)
        for cand_ligand in target_pool:
            if cand_ligand == true_ligand:
                continue
                
            sim = sim_matrix[true_ligand][cand_ligand]
            if 0.3 < sim < 0.8:
                # Crucial: Check if this "negative" is actually a known positive!
                if (p_id, cand_ligand) not in positive_edges:
                    selected_negative = cand_ligand
                    is_hard = True
                    hard_negative_count += 1
                    break
                    
        # Pass 2: Fallback to Easy Negative (similarity <= 0.3)
        if not selected_negative:
            for cand_ligand in target_pool:
                if cand_ligand == true_ligand:
                    continue
                if (p_id, cand_ligand) not in positive_edges:
                    selected_negative = cand_ligand
                    easy_negative_count += 1
                    break
                    
        # If we somehow STILL don't have a negative (extremely rare), skip to avoid failing
        if not selected_negative:
            continue
            
        # Create the negative row
        neg_row = row.copy()
        neg_row['Single_SMILES'] = selected_negative
        neg_row['Label'] = 0  # Y=0
        negative_records.append(neg_row)
        
    # 5. Compile Final Dataset
    print(f"\nSampling Complete:")
    print(f"  Hard Negatives Generated: {hard_negative_count}")
    print(f"  Easy Negatives Generated: {easy_negative_count}")
    
    # Add Label=1 to the original positive dataframe
    df['Label'] = 1
    
    # Combine and shuffle
    final_df = pd.concat([df, pd.DataFrame(negative_records)], ignore_index=True)
    final_df = final_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    print(f"\nFinal Dataset Size: {len(final_df)} rows (1:1 Ratio)")
    return final_df

# --- Usage Example ---
# Assuming 'exploded_df' is the output from Step 1
# balanced_df = generate_negative_samples(exploded_df)
# balanced_df.to_csv('final_training_data.csv', index=False)

In [ ]:

# df = pd.read_parquet("train_test_split_hard_cold.pq")
df = pd.read_parquet("dataset_164_step4.pq")

In [ ]:
df.head(1)

In [ ]:
df.rename(columns = {
    "Valid_SMILES": "Single_SMILES"
}, inplace=True
         )

In [ ]:
final_df = generate_negative_samples(df)

In [ ]:
final_df.shape

In [ ]:
final_df.head(1)

In [ ]:
final_df

In [ ]:
final_df.columns

In [ ]:
final_df = final_df.drop(columns=["Unnamed: 0"], axis=0)

In [ ]:
final_df.to_parquet("training_data_164.pq")

## For moltrans

In [ ]:
import gzip
import shutil
import subprocess
import pandas as pd

In [ ]:
df = pd.read_parquet("training_data.pq")

In [ ]:
df.head(1)

In [ ]:
def extract_sequence_from_cif(cif_gz_path, uid, tracker):
    """Extracts the 1D amino acid sequence from an AlphaFold .cif.gz file."""
    seq = []
    try:
        with gzip.open(cif_gz_path, 'rt') as f:
            capture = False
            for line in f:
                if line.startswith('_entity_poly.pdbx_seq_one_letter_code_can'):
                    parts = line.split()
                    if len(parts) > 1:
                        return parts[1].strip(';')
                    capture = True
                elif capture:
                    if line.startswith(';'):
                        if seq: break
                        seq.append(line.strip('; \n'))
                    elif line.startswith('_'):
                        break
                    else:
                        seq.append(line.strip())
            
            final_seq = "".join(seq).replace('\n', '').replace(' ', '')
            if not final_seq:
                tracker.append((uid, 'Sequence Extraction', 'Sequence tag found but empty'))
            return final_seq
            
    except Exception as e:
        tracker.append((uid, 'Sequence Extraction', f'File Read Error: {str(e)}'))
        return None

In [ ]:
protein_to_seq = dict()
tracker = []
for protein in (df["UniProt_ID"]):
    if protein not in protein_to_seq:
        protein_to_seq[protein] = extract_sequence_from_cif(f"alphafold_cif/AF-{protein}.cif.gz", protein, tracker)

In [ ]:
df["UniProt_ID"].nunique()

In [ ]:
len(protein_to_seq)

In [ ]:
columns = ["SMILES", "Target Sequence", "Label"]

In [ ]:
df["Target Sequence"] = df["UniProt_ID"].apply(lambda x: protein_to_seq[x])

In [ ]:
df["SMILES"] = df["Single_SMILES"]

In [ ]:
df["Dataset_Split"].value_counts(normalize = True)

In [ ]:
path = "/Users/aadi/Documents/Study material/MolTrans/dataset/metabolic_pathway/"

In [ ]:
df[df["Dataset_Split"] == "strict_train"][columns].reset_index(drop=True).to_csv(f"{path}/train.csv")

In [ ]:
x = df[df["Dataset_Split"] == "strict_test"][columns]

In [ ]:
x = x.sample(frac = 1)
val = x[:x.shape[0] // 2].reset_index(drop=True)
test = x[x.shape[0] // 2:].reset_index(drop=True)

In [ ]:
val.to_csv(f"{path}/val.csv")
test.to_csv(f"{path}/test.csv")

In [ ]:
import torch

In [ ]:
protein_embeddings = torch.load("protein_embeddings.pt")

In [ ]:
for key in protein_embeddings:
    if key not in protein_to_seq:
        print(key)

In [ ]:
protein_embeddings_seq = dict()
for key, value in protein_embeddings.items():
    seq = protein_to_seq[key]
    protein_embeddings_seq[seq] = value

In [ ]:
protein_embeddings_seq["AATPSWSAVPSMGTPRVPARTVLFQRERTGLTYRVPALLFAPPGPTLLAFAEQRLSPDDAHAHRLVQRTGTLAGGSVRWGAPRVLGTAALDEHRSMNPCPVHDERTATVFLFFIAVRGRTPEAAQIASGRNAARLCCVTSRDAGRSWGGARDLTAEAVGSAEQDWATFAVGPGHGVQLRSGRLLVPAYTYHVVRRECFGRICRTRPQSFAFYSDDHGRTWQRGGLVPSLRSGECQLAAVDGGRAGGVLYCNARSPLGSRVQALSVDEGTSFLPGELVPALAETARGCQGSIVGFPAPPASGWKDEGWSAGTSNPLRLPHLCPGAREAPEEGTGDTRGGGGLGGMEGCGNGPREPGENWGSWASVLLGPLAALSQSPTWLLYSHPVGRRARLHMGVRLSRSPLDPHSWTEPWVIHQGPSGYSDLASIGPAPGGVPTFACLFESGARVSYEEISFCMFSLQEVLENVRLGGVHPGPGDKPGDKPAGRCQPS"].shape

In [ ]:
torch.save("protein_embeddings_seq", f"{path}/")

In [ ]:
df = pd.read_parquet("training_data_164.pq")

In [ ]:
df.shape

In [ ]:
df["Single_SMILES"].nunique()

In [ ]:
df.groupby("Dataset_Split").agg({"Ligand": "nunique", "UniProt_ID": "nunique"}).tail(2)

In [ ]:
1811 + 436

In [ ]:
	Ligand	UniProt_ID
Dataset_Split		
strict_test	436	4668
strict_train	1811	38723
